### Preparando o ambiente

In [104]:
import pandas as pd
from imblearn.over_sampling import SMOTE
from sklearn.model_selection import train_test_split
from sklearn.neural_network import MLPClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report

In [105]:
df_original = pd.read_csv('data/radiomic_data_binary.csv')
df_preprocessed = pd.read_csv('data/radiomic_data_scaled.csv')
df_preprocessed_pca = pd.read_csv('data/radiomic_data_preprocessed_pca.csv')

In [106]:
df_original.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2018 entries, 0 to 2017
Columns: 116 entries, diagnostics_Versions_PyRadiomics to class
dtypes: float64(96), int64(2), object(18)
memory usage: 1.8+ MB


In [107]:
df_preprocessed.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2018 entries, 0 to 2017
Data columns (total 96 columns):
 #   Column                                              Non-Null Count  Dtype  
---  ------                                              --------------  -----  
 0   diagnostics_Image-original_Mean                     2018 non-null   float64
 1   diagnostics_Mask-original_VoxelNum                  2018 non-null   float64
 2   diagnostics_Mask-original_VolumeNum                 2018 non-null   float64
 3   original_firstorder_10Percentile                    2018 non-null   float64
 4   original_firstorder_90Percentile                    2018 non-null   float64
 5   original_firstorder_Energy                          2018 non-null   float64
 6   original_firstorder_Entropy                         2018 non-null   float64
 7   original_firstorder_InterquartileRange              2018 non-null   float64
 8   original_firstorder_Kurtosis                        2018 non-null   float64
 9

In [108]:
df_preprocessed_pca.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2018 entries, 0 to 2017
Data columns (total 11 columns):
 #   Column  Non-Null Count  Dtype  
---  ------  --------------  -----  
 0   PC1     2018 non-null   float64
 1   PC2     2018 non-null   float64
 2   PC3     2018 non-null   float64
 3   PC4     2018 non-null   float64
 4   PC5     2018 non-null   float64
 5   PC6     2018 non-null   float64
 6   PC7     2018 non-null   float64
 7   PC8     2018 non-null   float64
 8   PC9     2018 non-null   float64
 9   PC10    2018 non-null   float64
 10  class   2018 non-null   object 
dtypes: float64(10), object(1)
memory usage: 173.6+ KB


In [ ]:
# Checando a proporção de classes no dataset original
df_original['class'].value_counts()

class
BENIGN       1100
MALIGNANT     918
Name: count, dtype: int64

## Analisando o desempenho da árvore de decisão com os 2 conjuntos de dados

### Dados originais

In [110]:
# Removendo colunas com valores iguais em todo o dataset e identificadores
for col in df_original.columns:
    if df_original[col].nunique() == 1:
        df_original.drop(col, axis=1, inplace=True)

identifiers = [
    'diagnostics_Image-original_Hash',
    'diagnostics_Mask-original_Hash',
    'diagnostics_Mask-original_BoundingBox',
    'diagnostics_Mask-original_CenterOfMassIndex',
    'diagnostics_Mask-original_CenterOfMass'
]

df_original.drop(identifiers, axis=1, inplace=True)

In [111]:
X_original = df_original.drop(columns=['class'])
y_original = df_original['class']

X_train, X_test, y_train, y_test = train_test_split(X_original, y_original, test_size=0.2, random_state=42)

model = DecisionTreeClassifier(random_state=42)
model.fit(X_train, y_train)

y_pred = model.predict(X_test)

print(classification_report(y_test, y_pred))

              precision    recall  f1-score   support

      BENIGN       0.60      0.64      0.62       216
   MALIGNANT       0.55      0.50      0.52       188

    accuracy                           0.58       404
   macro avg       0.57      0.57      0.57       404
weighted avg       0.57      0.58      0.57       404



### Dados pré-processados

In [112]:
X_preprocessed = df_preprocessed.drop(columns=['class'])
y_preprocessed = df_preprocessed['class']

X_train, X_test, y_train, y_test = train_test_split(X_preprocessed, y_preprocessed, test_size=0.2, random_state=42)

model = DecisionTreeClassifier(random_state=42)
model.fit(X_train, y_train)

y_pred = model.predict(X_test)

print(classification_report(y_test, y_pred))

              precision    recall  f1-score   support

      BENIGN       0.58      0.63      0.60       216
   MALIGNANT       0.53      0.47      0.50       188

    accuracy                           0.56       404
   macro avg       0.55      0.55      0.55       404
weighted avg       0.55      0.56      0.55       404



### Dados pré-processados com PCA

In [120]:
X_preprocessed = df_preprocessed_pca.drop(columns=['class'])
y_preprocessed = df_preprocessed_pca['class']

X_train, X_test, y_train, y_test = train_test_split(X_preprocessed, y_preprocessed, test_size=0.2, random_state=42)

model = DecisionTreeClassifier(random_state=42)
model.fit(X_train, y_train)

y_pred = model.predict(X_test)

print(classification_report(y_test, y_pred))

              precision    recall  f1-score   support

      BENIGN       0.59      0.58      0.58       216
   MALIGNANT       0.52      0.53      0.53       188

    accuracy                           0.56       404
   macro avg       0.56      0.56      0.56       404
weighted avg       0.56      0.56      0.56       404



### Análise dos resultados aplicando árvore de decisão
Houve uma melhora nos resultados ao aplicar árvore de decisão, com uma acurácia de 55% para os dados pré-processados, 57% para os dados originais e 56% para os dados pré-processados com PCA.

Analisando o F1 score de todos está por volta de 50%, o que significa que o modelo está tendo problemas em prever ambas as classes. 

Esse resultado não é satisfatório, porém, isso pode estar acontecendo devido à natureza do modelo. Por ser simples, ele não consegue capturar padrões complexos nos dados.

## Agora vamos analisar os resultados aplicando MLP

### Dados originais

In [116]:
X_original = df_original.drop(columns=['class'])
y_original = df_original['class']

X_train, X_test, y_train, y_test = train_test_split(X_original, y_original, test_size=0.2, random_state=42)

# Utilizando MLP
model = MLPClassifier(alpha=1e-4, hidden_layer_sizes=(150, 100, 50), max_iter=300, random_state=42)
model.fit(X_train, y_train)

y_pred = model.predict(X_test)

print(classification_report(y_test, y_pred))

              precision    recall  f1-score   support

      BENIGN       0.00      0.00      0.00       216
   MALIGNANT       0.47      1.00      0.64       188

    accuracy                           0.47       404
   macro avg       0.23      0.50      0.32       404
weighted avg       0.22      0.47      0.30       404



c:\Users\Administrator\Documents\prova_2_paradigmas_ml\.venv\Lib\site-packages\sklearn\metrics\_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
c:\Users\Administrator\Documents\prova_2_paradigmas_ml\.venv\Lib\site-packages\sklearn\metrics\_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
c:\Users\Administrator\Documents\prova_2_paradigmas_ml\.venv\Lib\site-packages\sklearn\metrics\_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _wa

### Dados pré-processados

In [114]:
X_original = df_preprocessed.drop(columns=['class'])
y_original = df_preprocessed['class']

X_train, X_test, y_train, y_test = train_test_split(X_original, y_original, test_size=0.2, random_state=42)

# Utilizando MLP
model = MLPClassifier(alpha=1e-4, hidden_layer_sizes=(150, 100, 50), max_iter=300, random_state=42)
model.fit(X_train, y_train)

y_pred = model.predict(X_test)

print(classification_report(y_test, y_pred))

              precision    recall  f1-score   support

      BENIGN       0.62      0.69      0.66       216
   MALIGNANT       0.60      0.52      0.56       188

    accuracy                           0.61       404
   macro avg       0.61      0.61      0.61       404
weighted avg       0.61      0.61      0.61       404



### Dados preprocessados com PCA

In [115]:
X_preprocessed = df_preprocessed_pca.drop(columns=['class'])
y_preprocessed = df_preprocessed_pca['class']

X_train, X_test, y_train, y_test = train_test_split(X_preprocessed, y_preprocessed, test_size=0.2, random_state=42)

# Utilizando MLP
model = MLPClassifier(alpha=1e-4, hidden_layer_sizes=(150, 100, 50), max_iter=300, random_state=42)
model.fit(X_train, y_train)

y_pred = model.predict(X_test)

print(classification_report(y_test, y_pred))

              precision    recall  f1-score   support

      BENIGN       0.54      0.59      0.56       216
   MALIGNANT       0.47      0.41      0.44       188

    accuracy                           0.51       404
   macro avg       0.50      0.50      0.50       404
weighted avg       0.51      0.51      0.51       404



### Análise dos resultados aplicando árvore de decisão
Houve uma melhora nos resultados ao aplicar árvore de decisão, com uma acurácia de 61% para os dados pré-processados, 23% para os dados originais e 51% para os dados pré-processados com PCA.

## Final
Os modelos baseados em árvores de decisão são mais simples, por isso não são capazes de capturar padrões complexos nos dados.
Já os modelos baseados em perceptrons são mais complexos, e conseguem realizar análises mais profundas em conjuntos de dados com muitos atributos.

O desempenho dos modelos de MLP se sobressaiu devido à grande quantidade de atributos que os modelos tinham que processar.